# 08 — Doublet Calibrants: LaB₆ and Cu Kα₁/Kα₂

Most of the paper's headline numbers are on CeO₂ at high energy
(63–72 keV) — a calibrant chosen partly because its rings are
well-separated in 2θ.  Two regimes that *aren't* well-separated:

1. **LaB₆ at small L_sd** — adjacent rings (e.g., 200/210, 220/300)
   come within ~5 px of each other on a typical area detector at
   short L_sd.  The default singleton pseudo-Voigt fitter merges
   them and reports a biased centroid.

2. **Cu Kα radiation on a lab area detector** — every ring
   appears as a Kα₁ + Kα₂ doublet (Δλ/λ ≈ 2.5 × 10⁻³, intensity
   ratio 0.5).  At 2θ ≈ 30°, the doublet separation is ~3 px on a
   100 µm detector at 200 mm L_sd — close enough to merge.

For both cases the framework ships
[`forward.peak_fit_doublet`](../midas_calibrate_v2/forward/peak_fit_doublet.py),
a shared two-peak pseudo-Voigt LM that co-fits both peaks
simultaneously.  This notebook validates it on synthetic data and
quantifies the bias the singleton path produces.


In [1]:
import os, math
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import numpy as np
import torch

from midas_calibrate_v2.forward.peak_fit_batched import fit_cake_per_ring_batched
from midas_calibrate_v2.forward.peak_fit_doublet import fit_doublet_pairs

SQRT_2PI = math.sqrt(2.0 * math.pi)
INV_PI = 1.0 / math.pi

def pv_1d(R, c, sigma, gamma, eta_v, A):
    dR = R - c
    G = np.exp(-0.5 * dR*dR / (sigma*sigma)) / (sigma * SQRT_2PI)
    L = (gamma * INV_PI) / (dR*dR + gamma*gamma)
    return A * (eta_v * L + (1.0 - eta_v) * G)


## Cu Kα₁/Kα₂ within-ring doublet on synthetic LaB₆ (110)

Setup mimics a typical lab Cu-source area-detector geometry:
- λ(Kα₁) = 1.5406 Å, λ(Kα₂) = 1.5444 Å, I(Kα₂)/I(Kα₁) = 0.50
- LaB₆ a = 4.157 Å, (110) ring at 2θ ≈ 30.4° for Kα₁
- L_sd = 200 mm, p_x = 100 µm
- 2θ separation ≈ 0.077°, R separation ≈ 3.6 px


In [2]:
lam_a1 = 1.5406; lam_a2 = 1.5444; I_ratio = 0.50
a_LaB6 = 4.156826; hkl_norm = math.sqrt(2.0)         # (110)
Lsd_um = 200_000.0; px_um = 100.0
sigma_truth, gamma_truth, eta_v_truth, A_truth = 1.5, 1.0, 0.4, 200.0
n_eta = 90

tt_a1 = 2.0 * math.degrees(math.asin(lam_a1 * hkl_norm / (2.0 * a_LaB6)))
tt_a2 = 2.0 * math.degrees(math.asin(lam_a2 * hkl_norm / (2.0 * a_LaB6)))
R_a1 = Lsd_um * math.tan(math.radians(tt_a1)) / px_um
R_a2 = Lsd_um * math.tan(math.radians(tt_a2)) / px_um
sep_R = R_a2 - R_a1
print(f'Kα₁: 2θ={tt_a1:.4f}°  R={R_a1:.2f} px')
print(f'Kα₂: 2θ={tt_a2:.4f}°  R={R_a2:.2f} px')
print(f'Δ(2θ)={tt_a2-tt_a1:.4f}°  ΔR={sep_R:.2f} px (default doublet threshold = 25 px)')


Kα₁: 2θ=30.3855°  R=1172.71 px
Kα₂: 2θ=30.4623°  R=1176.32 px
Δ(2θ)=0.0768°  ΔR=3.60 px (default doublet threshold = 25 px)


## Build a synthetic cake with both peaks


In [3]:
rng = np.random.default_rng(0)
R_min, R_max, dR = R_a1 - 8.0, R_a2 + 8.0, 0.1
R_centers = np.arange(R_min, R_max + dR, dR)
n_R = len(R_centers)
eta_centers = np.linspace(-180.0, 180.0, n_eta, endpoint=False) + (360.0 / n_eta) / 2.0

cake = np.zeros((n_R, n_eta), dtype=np.float64)
for j in range(n_eta):
    I_lo = pv_1d(R_centers, R_a1, sigma_truth, gamma_truth, eta_v_truth, A_truth)
    I_hi = pv_1d(R_centers, R_a2, sigma_truth, gamma_truth, eta_v_truth, A_truth * I_ratio)
    cake[:, j] = I_lo + I_hi + 5.0 + rng.normal(0.0, 2.0, n_R)
print(f'cake shape: {cake.shape}')


cake shape: (198, 90)


## Singleton fit (the default) vs doublet co-fit


In [4]:
dt = torch.float64
cake_t = torch.tensor(cake, dtype=dt)
R_t = torch.tensor(R_centers, dtype=dt)
eta_t = torch.tensor(eta_centers, dtype=dt)

# Singleton: pretend there's only one ring at the Kα₁ position
rt_a1 = torch.tensor([R_a1], dtype=dt)
bf_singleton = fit_cake_per_ring_batched(
    cake_t, R_t, eta_t, rt_a1,
    half_window_px=8.0, max_iter=80, snr_min=2.0,
    dtype=dt, device='cpu', verbose=False,
)
R_singleton = bf_singleton.R_fit.cpu().numpy()
keep_s = (bf_singleton.rc.cpu().numpy() >= 0)

# Doublet co-fit: pass both peaks as a pair
rt_pair = torch.tensor([R_a1, R_a2], dtype=dt)
df = fit_doublet_pairs(
    cake_t, R_t, eta_t, rt_pair, pair_indices=[(0, 1)],
    half_window_px=8.0, max_iter=80, snr_min=2.0,
    dtype=dt, device='cpu', verbose=False,
)
R_doublet_a1 = df.R_fit_lo.cpu().numpy()
R_doublet_a2 = df.R_fit_hi.cpu().numpy()
keep_d = (df.rc.cpu().numpy() >= 0)

bias_singleton = float(np.median(R_singleton[keep_s]) - R_a1)
bias_doublet_a1 = float(np.median(R_doublet_a1[keep_d]) - R_a1)
bias_doublet_a2 = float(np.median(R_doublet_a2[keep_d]) - R_a2)

print(f'\nKα₁ centroid recovery (truth = {R_a1:.4f} px):')
print(f'  Singleton (default):  {np.median(R_singleton[keep_s]):>10.4f} px  bias {bias_singleton:+.4f} px')
print(f'  Doublet co-fit:       {np.median(R_doublet_a1[keep_d]):>10.4f} px  bias {bias_doublet_a1:+.4f} px')
print(f'\nKα₂ centroid recovery (truth = {R_a2:.4f} px):')
print(f'  Doublet co-fit:       {np.median(R_doublet_a2[keep_d]):>10.4f} px  bias {bias_doublet_a2:+.4f} px')

print(f'\nPer-fit relative bias (would feed into LM as fake strain):')
print(f'  Singleton:    ΔR/R = {bias_singleton/R_a1*1e6:+.0f} ppm')
print(f'  Doublet:      ΔR/R = {bias_doublet_a1/R_a1*1e6:+.0f} ppm')
print(f'  improvement:  {abs(bias_singleton/bias_doublet_a1):.1f}×')



Kα₁ centroid recovery (truth = 1172.7148 px):
  Singleton (default):   1173.0102 px  bias +0.2954 px
  Doublet co-fit:        1172.7228 px  bias +0.0080 px

Kα₂ centroid recovery (truth = 1176.3184 px):
  Doublet co-fit:        1176.2871 px  bias -0.0313 px

Per-fit relative bias (would feed into LM as fake strain):
  Singleton:    ΔR/R = +252 ppm
  Doublet:      ΔR/R = +7 ppm
  improvement:  36.9×


## What this means for lab Cu-Kα calibration

The singleton fitter biases the recovered ring centroid by ~250 ppm
on this geometry — 250 µε of fake strain that the LM would attempt
to absorb into a geometry shift.  The doublet co-fit removes this
bias to <10 ppm.

To enable the doublet path in production, set
`doublet_separation_px` greater than the expected separation (in
this geometry, anything > 3.6 px):

```python
res = autocalibrate_pv(
    v1, image, spec=spec,
    n_iter=4, half_window_px=8.0,
    doublet_separation_px=10.0,  # ← enables the co-fit
    ...
)
```

The auto-pairing logic in
[`forward.doublets`](../midas_calibrate_v2/forward/doublets.py)
detects which ring pairs are within `doublet_separation_px` and
sends them to `fit_doublet_pairs`; singleton rings are still fit
by the standard pseudo-Voigt path.

## Same recipe for LaB₆ adjacent-ring doublets

For LaB₆ at small L_sd (high-energy synchrotron with short L_sd),
multiple ring pairs appear in the doublet regime: (200, 210),
(220, 300), etc.  The doublet machinery handles them automatically
when you set the threshold.  See
[`run_doublet_validation.py`](../dev/paper/runners/run_doublet_validation.py)
for a synthetic two-ring pair validation in that regime.
